# WordPress.org Data Analysis with DuckDB

This notebook demonstrates how to query WordPress.org data that has been extracted using Meltano and loaded into DuckDB.

**Important Note about Ratings**: WordPress.org uses a 0-100 percentage scale for plugin ratings, not the typical 1-5 star scale. A rating of 96% means 96% of reviewers gave the plugin a positive rating.

In [ ]:
import duckdb
import html

# Import data analysis and visualization libraries
try:
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    import plotly.express as px
    import plotly.graph_objects as go
    
    print("✅ All libraries imported successfully!")
    VISUALIZATION_AVAILABLE = True
except ImportError as e:
    print(f"⚠️  Some visualization libraries not available: {e}")
    print("Run: pip install pandas matplotlib seaborn plotly")
    VISUALIZATION_AVAILABLE = False

# Helper function to clean HTML entities in plugin names
def clean_plugin_name(name):
    """Clean HTML entities from plugin names for better display"""
    if name:
        return html.unescape(name)
    return name

# Connect to the DuckDB database
conn = duckdb.connect('../data/wordpress_data.duckdb')
print("✅ Connected to WordPress.org data database!")

## Explore Available Tables

Let's first see what tables are available in our database:

In [ ]:
# Show all tables in the database
tables = conn.execute("SHOW TABLES").fetchall()
print("Available tables:")
for table in tables:
    print(f"  - {table[0]}")

## Explore Available Data

Let's explore what data we have in each table:

In [ ]:
# Dynamically explore all available tables
tables = conn.execute("SHOW TABLES").fetchall()

for table_name, in tables:
    print(f"\n📊 Table: {table_name}")
    print("=" * 50)
    
    try:
        # Get table schema
        schema = conn.execute(f"DESCRIBE {table_name}").fetchall()
        print(f"Columns ({len(schema)}):")
        for col_name, col_type, *_ in schema[:10]:  # Show first 10 columns
            print(f"  - {col_name}: {col_type}")
        
        if len(schema) > 10:
            print(f"  ... and {len(schema) - 10} more columns")
        
        # Get record count
        count = conn.execute(f"SELECT COUNT(*) FROM {table_name}").fetchone()[0]
        print(f"\nRecord count: {count:,}")
        
        # Get sample data
        if count > 0:
            sample = conn.execute(f"SELECT * FROM {table_name} LIMIT 3").fetchdf()
            print(f"\nSample data:")
            display(sample.head())
        else:
            print("\nNo data available in this table.")
            
    except Exception as e:
        print(f"Error exploring table {table_name}: {e}")

print(f"\n✅ Found {len(tables)} table(s) in the database.")

## Extract More Data (Optional)

If you want to extract more data from WordPress.org (events, themes, stats), you can run the extraction directly from this notebook:

In [ ]:
# Run full data extraction (this may take several minutes)
# Uncomment the lines below to extract all available data streams

import subprocess
import os

def run_extraction(streams=None):
    """Run Meltano extraction with specified streams"""
    try:
        # Change to the parent directory where meltano.yml is located
        os.chdir('..')
        
        # Activate virtual environment and run extraction
        if streams:
            # Set specific streams
            cmd1 = ['bash', '-c', f"source ../venv/bin/activate && meltano config tap-wordpress-org set stream_selection '{streams}'"]
            result1 = subprocess.run(cmd1, capture_output=True, text=True)
            print(f"Stream selection result: {result1.stdout}")
            if result1.stderr:
                print(f"Stream selection errors: {result1.stderr}")
        
        # Run extraction
        cmd2 = ['bash', '-c', 'source ../venv/bin/activate && meltano el tap-wordpress-org target-duckdb']
        print(f"🔄 Starting extraction{'with streams: ' + str(streams) if streams else ' (all streams)'}...")
        print("This may take several minutes...")
        
        result2 = subprocess.run(cmd2, capture_output=True, text=True, timeout=300)  # 5 minute timeout
        
        if result2.returncode == 0:
            print("✅ Extraction completed successfully!")
            print("Reconnecting to database to see new data...")
            
            # Reconnect to database
            global conn
            conn.close()
            conn = duckdb.connect('../data/wordpress_data.duckdb')
            
            # Show updated table list
            tables = conn.execute("SHOW TABLES").fetchall()
            print(f"Updated tables: {[t[0] for t in tables]}")
            
        else:
            print(f"❌ Extraction failed with return code: {result2.returncode}")
            print(f"Error output: {result2.stderr}")
            
    except subprocess.TimeoutExpired:
        print("⏰ Extraction timed out after 5 minutes")
    except Exception as e:
        print(f"❌ Error running extraction: {e}")
    finally:
        # Change back to notebook directory
        os.chdir('notebook')

# Uncomment one of these lines to run extraction:

# Extract all available data (plugins, events, themes, stats) - SLOW
# run_extraction()

# Extract specific streams - FASTER
# run_extraction('["plugins", "events"]')  # Just plugins and events
# run_extraction('["themes"]')             # Just themes
# run_extraction('["stats"]')              # Just stats

print("💡 Tip: Uncomment one of the run_extraction() lines above to extract more data")
print("💡 Or use the terminal: cd .. && make extract-all")

## Data Analysis Examples

Let's perform some interesting queries on the WordPress.org data:

In [ ]:
# Count total number of plugins
try:
    plugin_count = conn.execute("SELECT COUNT(*) as total_plugins FROM plugins").fetchone()[0]
    print(f"Total number of plugins: {plugin_count:,}")
except Exception as e:
    print(f"Error counting plugins: {e}")

In [ ]:
# Find most popular plugins by active installations (if data is available)
try:
    popular_plugins = conn.execute("""
        SELECT DISTINCT name, active_installs, rating, num_ratings
        FROM plugins 
        WHERE active_installs IS NOT NULL 
        ORDER BY active_installs DESC, name
        LIMIT 10
    """).fetchdf()
    
    # Clean HTML entities in plugin names
    popular_plugins['name'] = popular_plugins['name'].apply(clean_plugin_name)
    
    print("Top 10 most popular plugins by active installations:")
    display(popular_plugins)
except Exception as e:
    print(f"Error finding popular plugins: {e}")

In [ ]:
# Analyze plugin ratings distribution (WordPress.org uses 0-100% scale)
try:
    rating_distribution = conn.execute("""
        SELECT 
            CASE 
                WHEN rating >= 95 THEN '95-100%'
                WHEN rating >= 90 THEN '90-95%'
                WHEN rating >= 85 THEN '85-90%'
                WHEN rating >= 80 THEN '80-85%'
                WHEN rating >= 70 THEN '70-80%'
                WHEN rating >= 60 THEN '60-70%'
                ELSE 'Below 60%'
            END as rating_range,
            COUNT(*) as plugin_count
        FROM plugins 
        WHERE rating IS NOT NULL
        GROUP BY rating_range
        ORDER BY rating_range DESC
    """).fetchdf()
    print("Plugin rating distribution (WordPress.org uses 0-100% scale):")
    display(rating_distribution)
except Exception as e:
    print(f"Error analyzing ratings: {e}")

In [ ]:
# Example: Custom query - find plugins with high ratings and many reviews
# Note: WordPress.org ratings are on 0-100% scale, so 90%+ is excellent
try:
    quality_plugins = conn.execute("""
        SELECT DISTINCT name, rating, num_ratings, active_installs
        FROM plugins 
        WHERE rating >= 90 
          AND num_ratings >= 100
        ORDER BY num_ratings DESC, name
        LIMIT 15
    """).fetchdf()
    
    # Clean HTML entities in plugin names
    quality_plugins['name'] = quality_plugins['name'].apply(clean_plugin_name)
    
    print("High-quality plugins (rating >= 90%, reviews >= 100):")
    display(quality_plugins)
except Exception as e:
    print(f"Error finding quality plugins: {e}")

## Visualizations

Let's create some visualizations of the WordPress.org data:

In [ ]:
# Create visualizations if libraries are available
if VISUALIZATION_AVAILABLE:
    try:
        # Plugin installation distribution visualization (more useful than ratings)
        install_data = conn.execute("""
            SELECT 
                CASE 
                    WHEN active_installs >= 5000000 THEN '5M+'
                    WHEN active_installs >= 1000000 THEN '1M-5M'
                    WHEN active_installs >= 100000 THEN '100K-1M'
                    WHEN active_installs >= 10000 THEN '10K-100K'
                    WHEN active_installs >= 1000 THEN '1K-10K'
                    WHEN active_installs > 0 THEN 'Under 1K'
                    ELSE 'Unknown'
                END as install_range,
                COUNT(*) as plugin_count
            FROM plugins 
            WHERE active_installs IS NOT NULL
            GROUP BY install_range
            ORDER BY 
                CASE 
                    WHEN install_range = '5M+' THEN 1
                    WHEN install_range = '1M-5M' THEN 2
                    WHEN install_range = '100K-1M' THEN 3
                    WHEN install_range = '10K-100K' THEN 4
                    WHEN install_range = '1K-10K' THEN 5
                    WHEN install_range = 'Under 1K' THEN 6
                    ELSE 7
                END
        """).fetchdf()
        
        if not install_data.empty:
            # Create bar plot
            plt.figure(figsize=(10, 6))
            bars = plt.bar(install_data['install_range'], install_data['plugin_count'])
            plt.title('WordPress Plugin Distribution by Installation Count')
            plt.xlabel('Active Installations')
            plt.ylabel('Number of Plugins')
            plt.xticks(rotation=45)
            
            # Add value labels on bars
            for bar in bars:
                height = bar.get_height()
                plt.text(bar.get_x() + bar.get_width()/2., height,
                        f'{int(height)}',
                        ha='center', va='bottom')
            
            plt.tight_layout()
            plt.show()
            
            # Create pie chart with Plotly
            fig = px.pie(install_data, values='plugin_count', names='install_range', 
                        title='Plugin Distribution by Installation Count')
            fig.show()
        else:
            print("No plugin installation data available for visualization")
            
    except Exception as e:
        print(f"Error creating visualizations: {e}")
else:
    print("Visualization libraries not available. Install them with:")
    print("pip install pandas matplotlib seaborn plotly")

In [ ]:
# Popular plugins visualization
if VISUALIZATION_AVAILABLE:
    try:
        # Get top plugins by active installs (distinct to avoid duplicates)
        popular_plugins = conn.execute("""
            SELECT DISTINCT name, active_installs, rating
            FROM plugins 
            WHERE active_installs IS NOT NULL 
            ORDER BY active_installs DESC, name
            LIMIT 15
        """).fetchdf()
        
        if not popular_plugins.empty:
            # Clean HTML entities in plugin names
            popular_plugins['name'] = popular_plugins['name'].apply(clean_plugin_name)
            
            # Create horizontal bar chart
            plt.figure(figsize=(12, 8))
            plt.barh(popular_plugins['name'], popular_plugins['active_installs'])
            plt.title('Top 15 WordPress Plugins by Active Installations')
            plt.xlabel('Active Installations')
            plt.ylabel('Plugin Name')
            plt.gca().invert_yaxis()  # Show highest at top
            plt.tight_layout()
            plt.show()
            
            # Create scatter plot of rating vs active installs
            fig = px.scatter(popular_plugins, 
                           x='active_installs', 
                           y='rating',
                           hover_data=['name'],
                           title='Plugin Rating vs Active Installations',
                           labels={'active_installs': 'Active Installations', 'rating': 'Rating'})
            fig.show()
        else:
            print("No plugin installation data available for visualization")
            
    except Exception as e:
        print(f"Error creating popular plugins visualization: {e}")
else:
    print("Visualization libraries not available.")

In [ ]:
# Close the database connection
conn.close()
print("Database connection closed.")

## Next Steps

This notebook provides a starting point for analyzing WordPress.org data. You can:

1. **Extract more data**: Use the extraction cell above or run `make extract-all` in the terminal
2. **Create custom visualizations**: Use matplotlib, seaborn, or plotly to create your own charts
3. **Export results**: Save interesting findings to CSV or other formats
4. **Build dashboards**: Create interactive dashboards using tools like Streamlit
5. **Set up automation**: Use Meltano schedules to keep your data fresh

## Quick Commands

From the terminal (in the project directory):
```bash
# Get fresh sample data
make sample-data

# Extract all data streams (takes longer)
make extract-all  

# Extract specific data
make extract-plugins
make extract-events
make extract-themes

# Check what data you have
make check-data

# See all available commands
make help
```

## Database Connection

The database connection will automatically close when this notebook ends, but you can reconnect anytime:
```python
import duckdb
conn = duckdb.connect('../data/wordpress_data.duckdb')
```